In [19]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [20]:
load_dotenv()

True

In [21]:
class QuadState(TypedDict):
    a: float
    b: float
    c: float

    equation: str
    discriminant: float
    result: str

In [22]:
def show_equation(state: QuadState):
    equation = f' {state["a"]}x2 + {state["b"]}x + {state["c"]} = 0'
    return {"equation": equation}

In [23]:
def calculate_discriminant(state: QuadState):
    discriminant = state["b"] ** 2 - 4 * state["a"] * state["c"]
    return {"discriminant": discriminant}

In [24]:
def real_roots(state: QuadState):
    root1 = (-state["b"] + state["discriminant"] ** 0.5) / (2 * state["a"])
    root2 = (-state["b"] - state["discriminant"] ** 0.5) / (2 * state["a"])
    result = f'root 1  is {root1} and root 2 is {root2}'

    return {"result": result}

In [25]:
def repeated_roots(state: QuadState):
    root1 = (-state["b"] / (2 * state["a"]))
    result = f'root 1  is {root1} '
    return {"result": result}

In [26]:
def no_real_roots(state: QuadState):
    return {"result": f"no real roots"}

In [30]:
# conditionall function
from typing import Literal


def check_condition(State: QuadState) -> Literal["real_roots", "repeated_roots", "no_real_roots"]:
    if State["discriminant"] > 0:
        return "real_roots"
    elif State["discriminant"] == 0:
        return "repeated_roots"
    else:
        return "no_real_roots"

In [32]:
graph = StateGraph(QuadState)
graph.add_node('show_equation', show_equation)
graph.add_node('calculate_discriminant', calculate_discriminant)

graph.add_node('real_roots', real_roots)
graph.add_node('repeated_roots', repeated_roots)
graph.add_node('no_real_roots', no_real_roots)



graph.add_edge(START, 'show_equation')
graph.add_edge('show_equation', 'calculate_discriminant')

graph.add_conditional_edges('calculate_discriminant', check_condition)
graph.add_edge('calculate_discriminant', END)
graph.add_edge('real_roots', END)
graph.add_edge('repeated_roots', END)
graph.add_edge('no_real_roots', END)

workflow = graph.compile()
workflow.invoke({"a": 1, "b": 2, "c": 1})

{'a': 1,
 'b': 2,
 'c': 1,
 'equation': ' 1x2 + 2x + 1 = 0',
 'discriminant': 0,
 'result': 'root 1  is -1.0 '}

In [ ]:
f